# Coral Reef AI — RF-DETR Large + ConvNeXt-Small Full Pipeline

**Pipeline:** `Image/Video → RF-DETR Large → Bounding Box → +10% Context → ConvNeXt-Small (288×288) → Species → End-to-End / Video`

This notebook is designed as the high-capacity experiment. It keeps the project split intact, uses the detector as a single `target_coral` class, trains ConvNeXt-Small on the six operational coral labels, evaluates the integrated pipeline on validation, then runs all videos in `coral_video.zip` with temporal smoothing and high-quality H.264 output.

**Important:** detector held-out test is not used for model selection in this notebook.


In [ ]:
# CELL 1 — INSTALL + ENVIRONMENT
!pip -q install "rfdetr[train,loggers]==1.6.2" timm==1.0.22 scikit-learn pycocotools

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
import os, gc, csv, json, math, time, shutil, zipfile, random, subprocess
import numpy as np, pandas as pd, cv2
from PIL import Image
import torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from rfdetr import RFDETRLarge, RFDETR

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required')
DEVICE=torch.device('cuda')
GPU_NAME=torch.cuda.get_device_name(0)
VRAM_GB=torch.cuda.get_device_properties(0).total_memory/(1024**3)

if VRAM_GB>=70: RF_BATCH,RF_ACCUM,CLS_BATCH=16,1,128
elif VRAM_GB>=38: RF_BATCH,RF_ACCUM,CLS_BATCH=8,2,64
elif VRAM_GB>=22: RF_BATCH,RF_ACCUM,CLS_BATCH=4,4,32
else: RF_BATCH,RF_ACCUM,CLS_BATCH=2,8,16

print('GPU:',GPU_NAME, f'VRAM={VRAM_GB:.1f}GB')
print('RF batch/accum/effective:',RF_BATCH,RF_ACCUM,RF_BATCH*RF_ACCUM)
print('Classifier batch:',CLS_BATCH)
print('STATUS: ENVIRONMENT_READY')


In [ ]:
# CELL 2 — PATHS + RF-DETR COCO ADAPTER
PROJECT_ROOT=Path('/content/drive/MyDrive/Coral_Reef_AI_Project')
DET_ROOT=PROJECT_ROOT/'01_Data'/'Final'/'Detection_v1'
TRAIN_IMG=DET_ROOT/'images'/'train'; VAL_IMG=DET_ROOT/'images'/'validation'; TEST_IMG=DET_ROOT/'images'/'test'
TRAIN_COCO=DET_ROOT/'annotations'/'instances_train.json'; VAL_COCO=DET_ROOT/'annotations'/'instances_validation.json'
MANIFEST=PROJECT_ROOT/'04_Dataset_Splits'/'classification_roi_manifest_split.csv'
EXP_ROOT=PROJECT_ROOT/'06_Experiments'/'RFDETR_Large_ConvNeXtSmall_FullPipeline'
RF_OUT=EXP_ROOT/'01_RFDETR_Large'; CLS_OUT=EXP_ROOT/'02_ConvNeXtSmall'; E2E_OUT=EXP_ROOT/'03_EndToEnd'; VIDEO_OUT=EXP_ROOT/'04_Video'
for d in [RF_OUT,CLS_OUT,E2E_OUT,VIDEO_OUT]: d.mkdir(parents=True,exist_ok=True)
for p in [TRAIN_IMG,VAL_IMG,TRAIN_COCO,VAL_COCO,MANIFEST]:
    if not p.exists(): raise FileNotFoundError(p)

RF_DATA=Path('/content/Coral_RFDETR_Large_Dataset')
if RF_DATA.exists(): shutil.rmtree(RF_DATA)
(RF_DATA/'train').mkdir(parents=True); (RF_DATA/'valid').mkdir(parents=True)

def prep_coco(coco_path, src_dir, out_dir):
    data=json.loads(coco_path.read_text(encoding='utf-8'))
    for im in data['images']:
        name=Path(im['file_name']).name; src=src_dir/name
        if not src.exists():
            m=list(src_dir.glob(Path(name).stem+'.*'))
            if not m: raise FileNotFoundError(name)
            src=m[0]; name=src.name
        shutil.copy2(src,out_dir/name); im['file_name']=name
    data['categories']=[{'id':1,'name':'target_coral','supercategory':'coral'}]
    for a in data['annotations']: a['category_id']=1
    (out_dir/'_annotations.coco.json').write_text(json.dumps(data,ensure_ascii=False),encoding='utf-8')
    return len(data['images']),len(data['annotations'])

tr_i,tr_b=prep_coco(TRAIN_COCO,TRAIN_IMG,RF_DATA/'train')
va_i,va_b=prep_coco(VAL_COCO,VAL_IMG,RF_DATA/'valid')
assert (tr_i,tr_b,va_i,va_b)==(452,1540,100,354)
print('RF-DETR adapter:',tr_i,tr_b,'| val:',va_i,va_b)
print('TEST not attached. STATUS: DATA_READY')


In [ ]:
# CELL 3 — TRAIN RF-DETR LARGE
RF_EPOCHS=50; RF_RESOLUTION=704
RF_LR=1e-4; RF_LR_ENCODER=5e-5
print('RF-DETR Large | res',RF_RESOLUTION,'epochs',RF_EPOCHS,'batch',RF_BATCH,'accum',RF_ACCUM)
rf_model=RFDETRLarge()
t0=time.time()
rf_model.train(
    dataset_dir=str(RF_DATA), output_dir=str(RF_OUT),
    epochs=RF_EPOCHS, batch_size=RF_BATCH, grad_accum_steps=RF_ACCUM,
    lr=RF_LR, lr_encoder=RF_LR_ENCODER, weight_decay=1e-4,
    resolution=RF_RESOLUTION, optimizer='adamw',
    lr_scheduler='cosine', lr_scheduler_kwargs={'min_factor':0.05}, warmup_epochs=1.0,
    use_ema=True, eval_ema_only=False, checkpoint_interval=5,
    early_stopping=True, early_stopping_patience=12, early_stopping_min_delta=0.001,
    early_stopping_use_ema=True, skip_best_epochs=2, eval_max_dets=500,
    seed=SEED, device='cuda', tensorboard=True, wandb=False, progress_bar='tqdm'
)
RF_BEST=RF_OUT/'checkpoint_best_total.pth'
if not RF_BEST.exists(): raise FileNotFoundError(RF_BEST)
print(f'Training minutes={(time.time()-t0)/60:.2f}')
print('BEST:',RF_BEST)
print('STATUS: RFDETR_TRAINED')


In [ ]:
# CELL 4 — RF-DETR VALIDATION
try: del rf_model
except: pass
gc.collect(); torch.cuda.empty_cache()
rf_best=RFDETR.from_checkpoint(str(RF_BEST))
rf_val_metrics=rf_best.evaluate(dataset_dir=str(RF_DATA),split='val',resolution=RF_RESOLUTION,batch_size=RF_BATCH,device='cuda')
print('RF-DETR Large validation:')
for k,v in rf_val_metrics.items(): print(k,':',v)
(RF_OUT/'validation_metrics.json').write_text(json.dumps({str(k):float(v) if isinstance(v,(float,int,np.floating,np.integer)) else str(v) for k,v in rf_val_metrics.items()},indent=2),encoding='utf-8')
print('STATUS: RFDETR_VALIDATED')


In [ ]:
# CELL 5 — BUILD CONVNEXT-SMALL ROI DATASET (+10% CONTEXT)
CONTEXT_PAD=0.10; CLS_SIZE=288
CLS_LOCAL=Path('/content/Coral_ConvNeXtSmall_288_Context')
if CLS_LOCAL.exists(): shutil.rmtree(CLS_LOCAL)
for s in ['train','validation']: (CLS_LOCAL/s).mkdir(parents=True,exist_ok=True)
manifest=pd.read_csv(MANIFEST)
required={'roi_id','image_stem','raw_detection_label','x1_px','y1_px','x2_px','y2_px','split'}
if required-set(manifest.columns): raise RuntimeError('Manifest columns missing: '+str(required-set(manifest.columns)))

def source_dir(split): return {'train':TRAIN_IMG,'validation':VAL_IMG,'test':TEST_IMG}[split]
def resolve_img(folder,stem):
    p=folder/f'{stem}.jpg'
    if p.exists(): return p
    m=list(folder.glob(stem+'.*'))
    if not m: raise FileNotFoundError(stem)
    return m[0]
def context_crop(im,box,pad=0.10):
    W,H=im.size; x1,y1,x2,y2=map(float,box); bw=max(1.,x2-x1); bh=max(1.,y2-y1)
    px,py=bw*pad,bh*pad
    return im.crop((max(0,int(math.floor(x1-px))),max(0,int(math.floor(y1-py))),min(W,int(math.ceil(x2+px))),min(H,int(math.ceil(y2+py)))))

rows=[]
for split in ['train','validation']:
    sdf=manifest[manifest['split'].astype(str)==split]
    for _,r in sdf.iterrows():
        cls=str(r['raw_detection_label']); outd=CLS_LOCAL/split/cls; outd.mkdir(parents=True,exist_ok=True)
        src=resolve_img(source_dir(split),str(r['image_stem']))
        with Image.open(src) as im:
            crop=context_crop(im.convert('RGB'),(r.x1_px,r.y1_px,r.x2_px,r.y2_px),CONTEXT_PAD)
            op=outd/f"{r['roi_id']}.jpg"; crop.save(op,'JPEG',quality=95,subsampling=0)
        rows.append({'split':split,'roi_id':str(r['roi_id']),'species':cls,'crop_path':str(op)})
crop_manifest=pd.DataFrame(rows); crop_manifest.to_csv(CLS_OUT/'crop_manifest.csv',index=False)
assert (crop_manifest['split']=='train').sum()==1540 and (crop_manifest['split']=='validation').sum()==354

MODEL_NAME='convnext_small.fb_in22k_ft_in1k'
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
train_tf=transforms.Compose([
    transforms.RandomResizedCrop(CLS_SIZE,scale=(0.90,1.0),ratio=(0.92,1.08),interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.1),
    transforms.RandomRotation(5,interpolation=InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=0.12,contrast=0.12,saturation=0.08,hue=0.03),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
val_tf=transforms.Compose([transforms.Resize((CLS_SIZE,CLS_SIZE),interpolation=InterpolationMode.BICUBIC),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
train_ds=datasets.ImageFolder(CLS_LOCAL/'train',transform=train_tf)
train_eval_ds=datasets.ImageFolder(CLS_LOCAL/'train',transform=val_tf)
val_ds=datasets.ImageFolder(CLS_LOCAL/'validation',transform=val_tf)
class_to_idx=dict(train_ds.class_to_idx); idx_to_class={v:k for k,v in class_to_idx.items()}; NCLS=len(class_to_idx)
assert NCLS==6 and class_to_idx==val_ds.class_to_idx
counts=np.bincount([y for _,y in train_ds.samples],minlength=NCLS).astype(np.float32)
w=np.sqrt(counts.max()/np.maximum(counts,1)); w=w/w.mean(); class_weights=torch.tensor(w,dtype=torch.float32,device=DEVICE)
train_loader=DataLoader(train_ds,batch_size=CLS_BATCH,shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
train_eval_loader=DataLoader(train_eval_ds,batch_size=CLS_BATCH,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(val_ds,batch_size=CLS_BATCH,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
print('Classes:',class_to_idx); print('Counts:',counts.astype(int).tolist()); print('STATUS: CONVNEXT_DATA_READY')


In [ ]:
# CELL 6 — CONVNEXT TRAINING FUNCTIONS + STAGE A/B
classifier=timm.create_model(MODEL_NAME,pretrained=True,num_classes=NCLS).to(DEVICE)
criterion=nn.CrossEntropyLoss(weight=class_weights,label_smoothing=0.05)
BEST_CLS=CLS_OUT/'best_convnext_small_macro_f1.pth'
best_f1=-1.; best_stage=''; best_epoch=0; history=[]

def evaluate_cls(model,loader):
    model.eval(); yt=[]; yp=[]; probs_all=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16): logits=model(x)
            probs=torch.softmax(logits.float(),1); pred=probs.argmax(1)
            yt.extend(y.cpu().tolist()); yp.extend(pred.cpu().tolist()); probs_all.extend(probs.cpu().numpy())
    yt=np.asarray(yt); yp=np.asarray(yp); probs_all=np.asarray(probs_all,dtype=np.float32)
    m={'accuracy':accuracy_score(yt,yp),'macro_precision':precision_score(yt,yp,average='macro',zero_division=0),'macro_recall':recall_score(yt,yp,average='macro',zero_division=0),'macro_f1':f1_score(yt,yp,average='macro',zero_division=0),'weighted_f1':f1_score(yt,yp,average='weighted',zero_division=0)}
    return m,yt,yp,probs_all

def save_if_best(model,m,stage,epoch):
    global best_f1,best_stage,best_epoch
    if m['macro_f1']>best_f1:
        best_f1=float(m['macro_f1']); best_stage=stage; best_epoch=int(epoch)
        torch.save({'model_state_dict':model.state_dict(),'model_name':MODEL_NAME,'class_to_idx':class_to_idx,'input_size':CLS_SIZE,'context_padding':CONTEXT_PAD,'mean':MEAN,'std':STD,'stage':stage,'epoch':epoch,'val_metrics':{k:float(v) for k,v in m.items()}},BEST_CLS)

def train_stage(model,loader,epochs,lr,stage):
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,epochs),eta_min=max(1e-7,lr*0.03))
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    for ep in range(1,epochs+1):
        model.train(); total=0.; seen=0
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16): loss=criterion(model(x),y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); scaler.step(opt); scaler.update()
            total+=float(loss.detach().cpu())*x.size(0); seen+=x.size(0)
        sch.step(); m,_,_,_=evaluate_cls(model,val_loader); save_if_best(model,m,stage,ep)
        row={'stage':stage,'epoch':ep,'train_loss':total/max(1,seen),'lr':opt.param_groups[0]['lr'],**{f'val_{k}':float(v) for k,v in m.items()}}; history.append(row)
        print(f"{stage} ep {ep:02d}/{epochs} | loss={row['train_loss']:.4f} | Acc={m['accuracy']:.4f} | MacroF1={m['macro_f1']:.4f}")

for p in classifier.parameters(): p.requires_grad=False
for p in classifier.head.parameters(): p.requires_grad=True
train_stage(classifier,train_loader,2,1e-3,'A_head_only')
for p in classifier.parameters(): p.requires_grad=True
train_stage(classifier,train_loader,15,5e-5,'B_full_finetune')
pd.DataFrame(history).to_csv(CLS_OUT/'training_history_AB.csv',index=False)
print('Best after A/B:',best_f1,best_stage,best_epoch)
print('STATUS: CONVNEXT_AB_COMPLETE')


In [ ]:
# CELL 7 — TRAIN-ONLY HARD MINING + STAGE C + FINAL VALIDATION
ckpt=torch.load(BEST_CLS,map_location=DEVICE,weights_only=False); classifier.load_state_dict(ckpt['model_state_dict'])
train_m,yt,yp,pr=evaluate_cls(classifier,train_eval_loader)
true_conf=pr[np.arange(len(yt)),yt]
sw=np.ones(len(yt),dtype=np.float32); low=true_conf<0.60; wrong=yp!=yt
sw[low]*=1.25; sw[wrong]*=1.45
soft={class_to_idx[x] for x in ['Sarcophyton','Sinularia','WavingHand'] if x in class_to_idx}
soft_conf=np.array([(int(t) in soft and int(p) in soft and int(t)!=int(p)) for t,p in zip(yt,yp)])
sw[soft_conf]*=1.35; sw=np.minimum(sw,2.5)
miner=pd.DataFrame({'path':[p for p,_ in train_eval_ds.samples],'true_idx':yt,'pred_idx':yp,'true_conf':true_conf,'wrong':wrong,'low_conf':low,'soft_confusion':soft_conf,'weight':sw}); miner.to_csv(CLS_OUT/'hard_mining.csv',index=False)
sampler=WeightedRandomSampler(torch.tensor(sw,dtype=torch.double),num_samples=len(sw),replacement=True)
hard_loader=DataLoader(train_ds,batch_size=CLS_BATCH,sampler=sampler,num_workers=2,pin_memory=True,persistent_workers=True)
for p in classifier.parameters(): p.requires_grad=True
train_stage(classifier,hard_loader,8,1e-5,'C_hard_mining')
pd.DataFrame(history).to_csv(CLS_OUT/'training_history_complete.csv',index=False)

final_ckpt=torch.load(BEST_CLS,map_location=DEVICE,weights_only=False); classifier.load_state_dict(final_ckpt['model_state_dict']); classifier.eval()
cls_val_metrics,cls_true,cls_pred,cls_probs=evaluate_cls(classifier,val_loader)
rep=pd.DataFrame(classification_report(cls_true,cls_pred,target_names=[idx_to_class[i] for i in range(NCLS)],zero_division=0,output_dict=True)).T
rep.to_csv(CLS_OUT/'validation_classification_report.csv')
np.savetxt(CLS_OUT/'validation_confusion_matrix.csv',confusion_matrix(cls_true,cls_pred),fmt='%d',delimiter=',')
(CLS_OUT/'validation_summary.json').write_text(json.dumps({k:float(v) for k,v in cls_val_metrics.items()},indent=2),encoding='utf-8')
print('FINAL CONVNEXT VALIDATION:',cls_val_metrics)
print(rep[['precision','recall','f1-score','support']])
print('Best:',BEST_CLS,'| stage',best_stage,'epoch',best_epoch)
print('STATUS: CONVNEXT_TRAINED_AND_VALIDATED')


In [ ]:
# CELL 8 — END-TO-END HELPERS
val_manifest=manifest[manifest['split'].astype(str)=='validation'].copy().reset_index(drop=True)
assert len(val_manifest)==354
val_gt={}
for stem,g in val_manifest.groupby('image_stem'):
    val_gt[str(stem)]=[{'roi_id':str(r.roi_id),'species':str(r.raw_detection_label),'bbox':np.array([r.x1_px,r.y1_px,r.x2_px,r.y2_px],dtype=np.float32)} for _,r in g.iterrows()]
val_paths=[resolve_img(VAL_IMG,s) for s in sorted(val_gt)]

def iou_matrix(a,b):
    a=np.asarray(a,dtype=np.float32).reshape(-1,4); b=np.asarray(b,dtype=np.float32).reshape(-1,4)
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),dtype=np.float32)
    lt=np.maximum(a[:,None,:2],b[None,:,:2]); rb=np.minimum(a[:,None,2:],b[None,:,2:]); wh=np.clip(rb-lt,0,None); inter=wh[...,0]*wh[...,1]
    aa=np.clip(a[:,2]-a[:,0],0,None)*np.clip(a[:,3]-a[:,1],0,None); bb=np.clip(b[:,2]-b[:,0],0,None)*np.clip(b[:,3]-b[:,1],0,None)
    return inter/np.clip(aa[:,None]+bb[None,:]-inter,1e-9,None)

def greedy_match(pb,gb,thr=0.5):
    pb=np.asarray(pb,dtype=np.float32).reshape(-1,4); gb=np.asarray(gb,dtype=np.float32).reshape(-1,4)
    if len(pb)==0: return [],[],list(range(len(gb)))
    if len(gb)==0: return [],list(range(len(pb))),[]
    im=iou_matrix(pb,gb); cand=[(float(im[p,g]),p,g) for p in range(len(pb)) for g in range(len(gb)) if im[p,g]>=thr]; cand.sort(reverse=True)
    up=set(); ug=set(); matches=[]
    for v,p,g in cand:
        if p in up or g in ug: continue
        up.add(p); ug.add(g); matches.append((p,g,v))
    return matches,[p for p in range(len(pb)) if p not in up],[g for g in range(len(gb)) if g not in ug]

def cls_probs(crops):
    if not crops: return np.empty((0,NCLS),dtype=np.float32)
    batch=torch.stack([val_tf(c) for c in crops]).to(DEVICE,non_blocking=True)
    with torch.no_grad():
        with torch.autocast('cuda',dtype=torch.float16): logits=classifier(batch)
        probs=torch.softmax(logits.float(),1).cpu().numpy().astype(np.float32)
    del batch,logits
    return probs

print('Validation images:',len(val_paths),'GT:',sum(len(x) for x in val_gt.values()))
print('STATUS: E2E_HELPERS_READY')


In [ ]:
# CELL 9 — DETECTOR CONFIDENCE SWEEP + INTEGRATED E2E VALIDATION
try: del rf_best
except: pass
gc.collect(); torch.cuda.empty_cache()
detector=RFDETR.from_checkpoint(str(RF_BEST))
raw_det={}
for i,p in enumerate(val_paths,1):
    with Image.open(p) as im: d=detector.predict(im.convert('RGB'),threshold=0.03,include_source_image=False)
    raw_det[p.stem]={'boxes':np.asarray(d.xyxy,dtype=np.float32).reshape(-1,4),'conf':np.asarray(d.confidence,dtype=np.float32).reshape(-1)}
    if i%20==0: print('Detector inference',i,'/100')

sweep=[]
for conf in np.round(np.arange(0.05,0.81,0.025),3):
    TP=FP=FN=0
    for stem,items in val_gt.items():
        gb=np.array([x['bbox'] for x in items],dtype=np.float32).reshape(-1,4); pred=raw_det[stem]; keep=pred['conf']>=conf
        m,up,ug=greedy_match(pred['boxes'][keep],gb,0.5); TP+=len(m); FP+=len(up); FN+=len(ug)
    P=TP/(TP+FP) if TP+FP else 0.; R=TP/(TP+FN) if TP+FN else 0.; F=2*P*R/(P+R) if P+R else 0.
    sweep.append({'confidence':float(conf),'TP':TP,'FP':FP,'FN':FN,'precision':P,'recall':R,'f1':F})
sweep_df=pd.DataFrame(sweep); best_det=sweep_df.loc[sweep_df.f1.idxmax()]; BEST_DET_CONF=float(best_det.confidence)
sweep_df.to_csv(E2E_OUT/'detector_confidence_sweep.csv',index=False)
print('Best detector operating point:\n',best_det)

# integrated evaluation
gt_rows=[]; pred_rows=[]
for p in val_paths:
    stem=p.stem; items=val_gt[stem]; pred=raw_det[stem]; keep=pred['conf']>=BEST_DET_CONF; pb=pred['boxes'][keep]; pc=pred['conf'][keep]
    gb=np.array([x['bbox'] for x in items],dtype=np.float32).reshape(-1,4); matches,unmatched_p,unmatched_g=greedy_match(pb,gb,0.5)
    with Image.open(p) as im:
        rgb=im.convert('RGB'); crops=[context_crop(rgb,b,CONTEXT_PAD) for b in pb]
    probs=cls_probs(crops); pidx=probs.argmax(1) if len(probs) else np.empty(0,dtype=np.int64); pconf=probs.max(1) if len(probs) else np.empty(0)
    mg={g:(pp,iou) for pp,g,iou in matches}; mp={pp:(g,iou) for pp,g,iou in matches}; all_iou=iou_matrix(pb,gb)
    for gi,gt in enumerate(items):
        if gi in mg:
            pi,iou=mg[gi]; ps=idx_to_class[int(pidx[pi])]; correct=ps==gt['species']; mode='CORRECT' if correct else 'CLASSIFICATION_ERROR'
            gt_rows.append({'image_stem':stem,'roi_id':gt['roi_id'],'true_species':gt['species'],'detected':True,'iou':float(iou),'det_conf':float(pc[pi]),'pred_species':ps,'cls_conf':float(pconf[pi]),'e2e_correct':correct,'failure_mode':mode})
        else:
            bi=float(all_iou[:,gi].max()) if len(pb) else 0.; mode='LOCALIZATION_ERROR' if 0.10<=bi<0.50 else 'DETECTOR_MISS'
            gt_rows.append({'image_stem':stem,'roi_id':gt['roi_id'],'true_species':gt['species'],'detected':False,'iou':bi,'det_conf':np.nan,'pred_species':None,'cls_conf':np.nan,'e2e_correct':False,'failure_mode':mode})
    for pi in range(len(pb)):
        if pi in mp:
            gi,iou=mp[pi]; ts=items[gi]['species']; ps=idx_to_class[int(pidx[pi])]; matched=True; correct=ps==ts
        else:
            ts=None; ps=idx_to_class[int(pidx[pi])]; matched=False; correct=False; iou=float(all_iou[pi].max()) if len(gb) else 0.
        pred_rows.append({'image_stem':stem,'pred_index':pi,'det_conf':float(pc[pi]),'pred_species':ps,'cls_conf':float(pconf[pi]),'matched':matched,'true_species':ts,'iou':float(iou),'e2e_correct':bool(matched and correct)})

gt_df=pd.DataFrame(gt_rows); pred_df=pd.DataFrame(pred_rows); TOTAL_GT=len(gt_df); TOTAL_PRED=len(pred_df); MATCHED=int(gt_df.detected.sum()); CORRECT=int(gt_df.e2e_correct.sum())
DET_P=MATCHED/TOTAL_PRED if TOTAL_PRED else 0.; DET_R=MATCHED/TOTAL_GT; CLS_MATCH=CORRECT/MATCHED if MATCHED else 0.; E2E_P=CORRECT/TOTAL_PRED if TOTAL_PRED else 0.; E2E_R=CORRECT/TOTAL_GT; E2E_F1=2*E2E_P*E2E_R/(E2E_P+E2E_R) if E2E_P+E2E_R else 0.
E2E_SUMMARY={'validation_images':100,'validation_gt':TOTAL_GT,'predictions':TOTAL_PRED,'matched_detections':MATCHED,'correct_detection_and_species':CORRECT,'detector_confidence':BEST_DET_CONF,'detector_precision':DET_P,'detector_recall':DET_R,'classifier_accuracy_on_matched':CLS_MATCH,'end_to_end_species_precision':E2E_P,'end_to_end_species_recall':E2E_R,'end_to_end_species_f1':E2E_F1,'classification_errors':int((gt_df.failure_mode=='CLASSIFICATION_ERROR').sum()),'localization_errors':int((gt_df.failure_mode=='LOCALIZATION_ERROR').sum()),'detector_misses':int((gt_df.failure_mode=='DETECTOR_MISS').sum())}
gt_df.to_csv(E2E_OUT/'gt_outcomes.csv',index=False); pred_df.to_csv(E2E_OUT/'prediction_outcomes.csv',index=False); (E2E_OUT/'summary.json').write_text(json.dumps(E2E_SUMMARY,indent=2),encoding='utf-8')
print('E2E SUMMARY:',json.dumps(E2E_SUMMARY,indent=2)); print('Failure modes:\n',gt_df.failure_mode.value_counts())
print('STATUS: END_TO_END_VALIDATION_COMPLETE')


In [ ]:
# CELL 10 — VIDEO SETUP + MODEL-AGNOSTIC TEMPORAL TRACKER
EXPECTED_ZIP=Path('/content/drive/MyDrive/Coral_video/coral_video.zip'); MYDRIVE=Path('/content/drive/MyDrive')
def find_zip():
    if EXPECTED_ZIP.exists(): return EXPECTED_ZIP
    x=list(MYDRIVE.rglob('coral_video.zip'))
    if x: return x[0]
    x=[p for p in MYDRIVE.rglob('*.zip') if 'coral' in p.name.lower() and 'video' in p.name.lower()]
    if x: return x[0]
    raise FileNotFoundError('coral_video.zip not found')
VIDEO_ZIP=find_zip(); VWORK=Path('/content/RFDETR_ConvNeXt_Video'); VIN=VWORK/'input'; VLOCAL=VWORK/'output'
if VWORK.exists(): shutil.rmtree(VWORK)
VIN.mkdir(parents=True); VLOCAL.mkdir(parents=True)
with zipfile.ZipFile(VIDEO_ZIP,'r') as z: z.extractall(VIN)
exts={'.mp4','.mov','.avi','.mkv','.m4v','.webm','.mpg','.mpeg'}; video_paths=sorted([p for p in VIN.rglob('*') if p.is_file() and p.suffix.lower() in exts])
if not video_paths: raise RuntimeError('No videos in ZIP')
VIDEO_DET_CONF=BEST_DET_CONF; TRACK_IOU=0.20; MIN_HITS=2; MAX_MISSED=2; STATE_MAX_AGE=20
BBOX_ALPHA=0.45; VEL_ALPHA=0.50; CLASS_ALPHA=0.25; UNKNOWN_CONF=0.55; UNKNOWN_MARGIN=0.10
BOX_THICKNESS=5; FONT_SCALE=0.85; TEXT_THICKNESS=2; VIDEO_CRF=12; FFMPEG_PRESET='slow'; MAX_FRAMES_PER_VIDEO=None

def clip_box(b,W,H):
    x1,y1,x2,y2=map(float,b); return np.array([max(0,min(x1,W-1)),max(0,min(y1,H-1)),max(0,min(x2,W-1)),max(0,min(y2,H-1))],dtype=np.float32)
def new_track(tid,b,fi,dc): return {'id':tid,'box':np.asarray(b,dtype=np.float32).copy(),'vel':np.zeros(4,np.float32),'prob':None,'species':None,'cls_conf':0.,'margin':0.,'det_conf':float(dc),'last_seen':fi,'hits':1,'missed':0}
def update_track(s,b,dc,fi):
    old=s['box'].copy(); b=np.asarray(b,np.float32); new=BBOX_ALPHA*b+(1-BBOX_ALPHA)*old; iv=new-old; s['vel']=(VEL_ALPHA*iv+(1-VEL_ALPHA)*s['vel']).astype(np.float32); s['box']=new.astype(np.float32); s['det_conf']=float(dc); s['last_seen']=fi; s['hits']+=1; s['missed']=0
def pred_box(s,ahead=1): return (s['box']+s['vel']*float(ahead)).astype(np.float32)
def update_class(s,p):
    p=np.asarray(p,np.float32); sm=p.copy() if s['prob'] is None else CLASS_ALPHA*p+(1-CLASS_ALPHA)*s['prob']; sm/=max(float(sm.sum()),1e-9); s['prob']=sm
    o=np.argsort(sm)[::-1]; s['species']=idx_to_class[int(o[0])]; s['cls_conf']=float(sm[o[0]]); s['margin']=float(sm[o[0]]-sm[o[1]])
def label_for(s):
    if s['cls_conf']<UNKNOWN_CONF: return 'Unknown Coral'
    if s['margin']<UNKNOWN_MARGIN: return 'Uncertain Coral'
    return s['species']
def associate(states,boxes):
    ids=list(states.keys())
    if not ids or len(boxes)==0: return [],list(range(len(ids))),list(range(len(boxes))),ids
    pred=np.stack([pred_box(states[t]) for t in ids]); im=iou_matrix(pred,boxes); rr,cc=linear_sum_assignment(1-im); m=[]; mt=set(); md=set()
    for r,c in zip(rr,cc):
        if im[r,c]<TRACK_IOU: continue
        m.append((r,c,float(im[r,c]))); mt.add(r); md.add(c)
    return m,[i for i in range(len(ids)) if i not in mt],[i for i in range(len(boxes)) if i not in md],ids

def draw_track(frame,s,b,held=False):
    H,W=frame.shape[:2]; b=clip_box(b,W,H); x1,y1,x2,y2=[int(round(x)) for x in b]; lab=label_for(s)
    color=(0,200,255) if held else ((0,165,255) if lab in {'Unknown Coral','Uncertain Coral'} else (0,255,0))
    cv2.rectangle(frame,(x1,y1),(x2,y2),color,BOX_THICKNESS)
    l1=f"ID {s['id']} | {lab}"; l2=f"Det {s['det_conf']:.2f} | Cls {s['cls_conf']:.2f}"; font=cv2.FONT_HERSHEY_SIMPLEX
    (w1,h1),_=cv2.getTextSize(l1,font,FONT_SCALE,TEXT_THICKNESS); (w2,h2),_=cv2.getTextSize(l2,font,FONT_SCALE,TEXT_THICKNESS); lw=max(w1,w2)+18; lh=h1+h2+24; yy=max(0,y1-lh)
    cv2.rectangle(frame,(x1,yy),(min(W-1,x1+lw),y1),color,-1); cv2.putText(frame,l1,(x1+8,yy+h1+6),font,FONT_SCALE,(0,0,0),TEXT_THICKNESS,cv2.LINE_AA); cv2.putText(frame,l2,(x1+8,yy+h1+h2+14),font,FONT_SCALE,(0,0,0),TEXT_THICKNESS,cv2.LINE_AA)

print('Videos:',[p.name for p in video_paths]); print('Video det conf:',VIDEO_DET_CONF); print('STATUS: VIDEO_TRACKER_READY')


In [ ]:
# CELL 11 — RUN ALL VIDEOS (RF-DETR LARGE + CONVNEXT-SMALL)
classifier.eval(); video_detector=RFDETR.from_checkpoint(str(RF_BEST)); video_summaries=[]
for vi,vp in enumerate(video_paths,1):
    print('\n'+'='*110); print(f'VIDEO {vi}/{len(video_paths)}:',vp.name)
    cap=cv2.VideoCapture(str(vp));
    if not cap.isOpened(): print('skip'); continue
    fps=float(cap.get(cv2.CAP_PROP_FPS)); fps=fps if np.isfinite(fps) and fps>0 else 25.; W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stem=vp.stem.replace(' ','_'); ld=VLOCAL/stem; ld.mkdir(parents=True,exist_ok=True); outv=ld/f'{stem}_RFDETRL_ConvNeXtS_HQ.mp4'; outcsv=ld/f'{stem}_tracks.csv'
    cmd=['ffmpeg','-y','-f','rawvideo','-vcodec','rawvideo','-pix_fmt','bgr24','-s',f'{W}x{H}','-r',f'{fps:.8f}','-i','pipe:0','-i',str(vp),'-map','0:v:0','-map','1:a:0?','-c:v','libx264','-preset',FFMPEG_PRESET,'-crf',str(VIDEO_CRF),'-pix_fmt','yuv420p','-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(outv)]
    ff=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    cf=open(outcsv,'w',newline='',encoding='utf-8'); cw=csv.DictWriter(cf,fieldnames=['frame','time_sec','track_id','source','x1','y1','x2','y2','det_conf','species','cls_conf','class_margin']); cw.writeheader()
    states={}; next_id=1; fi=-1; unique=set(); held_rows=0; det_ms=[]; cls_ms=[]; pipe_ms=[]
    while True:
        ok,frame=cap.read()
        if not ok: break
        fi+=1
        if MAX_FRAMES_PER_VIDEO is not None and fi>=MAX_FRAMES_PER_VIDEO: break
        ts=time.perf_counter(); td=time.perf_counter(); pil=Image.fromarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)); d=video_detector.predict(pil,threshold=VIDEO_DET_CONF,include_source_image=False); det_ms.append((time.perf_counter()-td)*1000)
        boxes=np.asarray(d.xyxy,dtype=np.float32).reshape(-1,4); conf=np.asarray(d.confidence,dtype=np.float32).reshape(-1)
        matches,ut,ud,ids=associate(states,boxes); current=set()
        for ti,di,_ in matches:
            tid=ids[ti]; update_track(states[tid],boxes[di],conf[di],fi); current.add(tid)
        for di in ud:
            tid=next_id; next_id+=1; states[tid]=new_track(tid,boxes[di],fi,conf[di]); current.add(tid); unique.add(tid)
        for tid in list(states):
            if tid in current: continue
            age=fi-states[tid]['last_seen']; states[tid]['missed']=age
            if age>STATE_MAX_AGE: del states[tid]
        tc=time.perf_counter(); crop_ids=[]; crops=[]
        for tid in current:
            b=clip_box(states[tid]['box'],W,H); crops.append(context_crop(pil,b,CONTEXT_PAD)); crop_ids.append(tid)
        pp=cls_probs(crops)
        for tid,p in zip(crop_ids,pp): update_class(states[tid],p)
        cls_ms.append((time.perf_counter()-tc)*1000)
        for tid in sorted(current):
            if tid not in states or states[tid]['hits']<MIN_HITS: continue
            s=states[tid]; draw_track(frame,s,s['box'],False); b=s['box']; cw.writerow({'frame':fi,'time_sec':fi/fps,'track_id':tid,'source':'DETECTED','x1':float(b[0]),'y1':float(b[1]),'x2':float(b[2]),'y2':float(b[3]),'det_conf':s['det_conf'],'species':label_for(s),'cls_conf':s['cls_conf'],'class_margin':s['margin']})
        for tid,s in list(states.items()):
            if tid in current or s['hits']<MIN_HITS: continue
            if 1<=s['missed']<=MAX_MISSED:
                b=pred_box(s,s['missed']); draw_track(frame,s,b,True); held_rows+=1; cw.writerow({'frame':fi,'time_sec':fi/fps,'track_id':tid,'source':'HELD','x1':float(b[0]),'y1':float(b[1]),'x2':float(b[2]),'y2':float(b[3]),'det_conf':s['det_conf'],'species':label_for(s),'cls_conf':s['cls_conf'],'class_margin':s['margin']})
        try: ff.stdin.write(frame.tobytes())
        except BrokenPipeError: raise RuntimeError(ff.stderr.read().decode('utf-8',errors='ignore')[-5000:])
        pipe_ms.append((time.perf_counter()-ts)*1000)
        if (fi+1)%100==0:
            r=float(np.mean(pipe_ms[-100:])); print(f'Frames={fi+1} | IDs={len(unique)} | active={len(states)} | held={held_rows} | {r:.1f}ms ~{1000/r:.2f}FPS')
    cap.release(); ff.stdin.close(); ff.wait(); err=ff.stderr.read().decode('utf-8',errors='ignore'); cf.close()
    if ff.returncode!=0: raise RuntimeError(err[-5000:])
    summary={'video':vp.name,'frames':fi+1,'unique_track_ids':len(unique),'held_rows':held_rows,'mean_detector_ms':float(np.mean(det_ms)) if det_ms else 0.,'mean_classifier_ms':float(np.mean(cls_ms)) if cls_ms else 0.,'mean_pipeline_ms':float(np.mean(pipe_ms)) if pipe_ms else 0.}
    summary['pipeline_fps']=1000/summary['mean_pipeline_ms'] if summary['mean_pipeline_ms'] else 0.; video_summaries.append(summary)
    dd=VIDEO_OUT/stem
    if dd.exists(): shutil.rmtree(dd)
    shutil.copytree(ld,dd); (dd/f'{stem}_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print('Done:',summary); gc.collect(); torch.cuda.empty_cache()
print('STATUS: VIDEO_TEST_COMPLETE')


In [ ]:
# CELL 12 — FINAL SUMMARY + PACKAGE
video_df=pd.DataFrame(video_summaries); video_df.to_csv(VIDEO_OUT/'all_video_summary.csv',index=False)
FINAL={'pipeline':'RF-DETR Large + 10% Context + ConvNeXt-Small','detector_checkpoint':str(RF_BEST),'detector_resolution':RF_RESOLUTION,'detector_validation':{str(k):float(v) if isinstance(v,(float,int,np.floating,np.integer)) else str(v) for k,v in rf_val_metrics.items()},'detector_confidence':BEST_DET_CONF,'classifier_checkpoint':str(BEST_CLS),'classifier_validation':{k:float(v) for k,v in cls_val_metrics.items()},'end_to_end_validation':E2E_SUMMARY,'videos_processed':len(video_summaries),'mean_video_fps':float(video_df.pipeline_fps.mean()) if len(video_df) else 0.,'detector_test_used':False}
(EXP_ROOT/'FINAL_PIPELINE_SUMMARY.json').write_text(json.dumps(FINAL,indent=2,ensure_ascii=False),encoding='utf-8')
zip_path=Path(shutil.make_archive('/content/RFDETR_Large_ConvNeXtSmall_Coral_Results','zip',root_dir=str(EXP_ROOT))); drive_zip=EXP_ROOT/zip_path.name; shutil.copy2(zip_path,drive_zip)
print('='*120); print('FULL PIPELINE COMPLETE'); print('RF-DETR best:',RF_BEST); print('ConvNeXt best:',BEST_CLS); print('Classifier Macro-F1:',cls_val_metrics['macro_f1']); print('E2E F1:',E2E_F1); print('Videos:',len(video_summaries)); print('Results ZIP:',drive_zip); print('Detector held-out TEST was NOT used.'); print('STATUS: FULL_PIPELINE_COMPLETE')


## Optional final held-out detector test

After the final detector/checkpoint/threshold is frozen, evaluate the untouched detector test split **once** in a separate final-test notebook. This notebook intentionally keeps it out of model selection.
